# DWD-Tageswerte Berlin: Download als CSV

Lädt Tageswerte (Produkt `KL`, Klimadaten täglich) vom [DWD Open Data Server](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/) für 2020 bis heute und speichert sie als CSV.

| DWD-Spalte | Name im CSV |
|---|---|
| `TNK` | `temp_min` (°C) |
| `TXK` | `temp_max` (°C) |
| `TMK` | `temp_durch` (°C) |
| `RSK` | `niederschlagsmenge` (mm) |
| `SDK` | `sonnenstunden` (h) |

**Ablauf**
1. Kandidaten-Stationen in Berlin prüfen: Sind alle fünf Größen im Zeitraum vorhanden?
2. Passende Station wählen
3. Daten laden (`historical` + `recent`) und als CSV speichern

In [1]:
import io
import re
import zipfile

import pandas as pd
import requests

BASIS_URL = 'https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl'
START = pd.Timestamp('2020-01-01')

SPALTEN = {
    'TNK': 'temp_min',
    'TXK': 'temp_max',
    'TMK': 'temp_durch',
    'RSK': 'niederschlagsmenge',
    'SDK': 'sonnenstunden',
}

# Berliner Stationen: ID -> Name
KANDIDATEN = {
    '00400': 'Berlin-Buch',
    '00403': 'Berlin-Dahlem (FU)',
    '00420': 'Berlin-Frohnau',
    '00430': 'Berlin-Marzahn',
    '00433': 'Berlin-Tempelhof',
}

In [2]:
def lade_station(station_id: str) -> pd.DataFrame | None:
    """Lädt historical + recent einer Station und gibt die Rohdaten ab START zurück."""
    teile = []
    for ordner, muster in [('historical', rf'tageswerte_KL_{station_id}_\d{{8}}_\d{{8}}_hist\.zip'),
                           ('recent', rf'tageswerte_KL_{station_id}_akt\.zip')]:
        listing = requests.get(f'{BASIS_URL}/{ordner}/', timeout=60).text
        treffer = re.search(muster, listing)
        if not treffer:
            continue
        r = requests.get(f'{BASIS_URL}/{ordner}/{treffer.group(0)}', timeout=120)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            name = next(n for n in z.namelist() if n.startswith('produkt_klima_tag_'))
            teile.append(pd.read_csv(z.open(name), sep=';', skipinitialspace=True, na_values=['-999']))
    if not teile:
        return None
    df = pd.concat(teile)
    df['datum'] = pd.to_datetime(df['MESS_DATUM'].astype(str), format='%Y%m%d')
    # recent überschreibt historical bei Überschneidung
    df = df.drop_duplicates('datum', keep='last').sort_values('datum')
    return df[df['datum'] >= START]


rohdaten = {sid: lade_station(sid) for sid in KANDIDATEN}

In [6]:
# Verfügbarkeitsprüfung: Anteil vorhandener Werte je Größe, bezogen auf alle Tage im Zeitraum (1.0 = vollständig)
ENDE = max(df['datum'].max() for df in rohdaten.values() if df is not None and not df.empty)
SOLL_TAGE = (ENDE - START).days + 1

zeilen = []
for sid, df in rohdaten.items():
    if df is None or df.empty:
        zeilen.append({'station_id': sid, 'name': KANDIDATEN[sid], 'tage': 0})
        continue
    eintrag = {'station_id': sid, 'name': KANDIDATEN[sid], 'tage': len(df),
               'von': df['datum'].min().date(), 'bis': df['datum'].max().date()}
    for roh, neu in SPALTEN.items():
        eintrag[neu] = round(df[roh].notna().sum() / SOLL_TAGE, 3)
    zeilen.append(eintrag)

verfuegbarkeit = pd.DataFrame(zeilen).set_index('station_id')
verfuegbarkeit

,name,tage,von,bis,temp_min,temp_max,temp_durch,niederschlagsmenge,sonnenstunden
station_id,,,,,,,,,
00400,Berlin-Buch,2497,2020-01-01,2026-11-01,1.000,1.000,1.000,1.000,0.000
00403,Berlin-Dahlem (FU),2483,2020-01-01,2026-11-01,0.994,0.994,0.994,0.994,0.994
00420,Berlin-Frohnau,1984,2020-01-01,2026-11-01,0.792,0.792,0.792,0.792,0.000
00430,Berlin-Marzahn,490,2020-01-01,2021-05-04,0.196,0.196,0.196,0.196,0.196
00433,Berlin-Tempelhof,2497,2020-01-01,2026-11-01,1.000,1.000,1.000,0.996,0.394


In [7]:
# Station wählen: höchste Abdeckung der schlechtesten der fünf Größen über den gesamten Zeitraum
spalten_neu = list(SPALTEN.values())
vollstaendigkeit = verfuegbarkeit[spalten_neu].min(axis=1)
STATION_ID = vollstaendigkeit.idxmax()
print(f'Gewählt: {STATION_ID} {KANDIDATEN[STATION_ID]} (schlechteste Größe: {vollstaendigkeit[STATION_ID]:.1%} der Tage vorhanden)')
assert vollstaendigkeit[STATION_ID] > 0.9, 'Keine Station liefert alle Größen vollständig'

Gewählt: 00403 Berlin-Dahlem (FU) (schlechteste Größe: 99.4% der Tage vorhanden)


In [8]:
daten = (rohdaten[STATION_ID]
         .rename(columns=SPALTEN)
         .set_index('datum')[spalten_neu])

dateiname = f'dwd_berlin_{STATION_ID}_{daten.index.min():%Y%m%d}_{daten.index.max():%Y%m%d}.csv'
daten.to_csv(dateiname)
print(f'{len(daten)} Tage gespeichert in {dateiname}')
daten.head()

2483 Tage gespeichert in dwd_berlin_00403_20200101_20261101.csv


,temp_min,temp_max,temp_durch,niederschlagsmenge,sonnenstunden
datum,,,,,
2020-01-01,-2.8,5.0,1.8,0.0,2.9
2020-01-02,-4.1,6.2,-0.3,0.0,7.5
2020-01-03,0.8,6.9,4.3,4.4,0.0
2020-01-04,1.7,5.6,4.1,3.0,0.0
2020-01-05,-1.4,3.6,1.4,0.0,2.1
